In [3]:

import json
from pathlib import Path

from web_recon_server import (
    scan_http_headers,
    enumerate_http_methods,
    analyze_cookie_security
)


# --- AUTHORIZED LAB SCOPE ---

ALLOWED_TARGETS = {"127.0.0.1"}
ALLOWED_PORTS = {3000}

TARGET = "127.0.0.1"
PORT = 3000


def check_scope(target, port):
    """
    Verify that the target is within the authorized lab scope.
    """

    if target not in ALLOWED_TARGETS:
        raise PermissionError(
            f"BLOCKED: Target {target} is outside the authorized scope."
        )

    if port not in ALLOWED_PORTS:
        raise PermissionError(
            f"BLOCKED: Port {port} is outside the authorized scope."
        )

    return True


# --- VALIDATE TARGET ---

check_scope(TARGET, PORT)

print("=" * 60)
print("AUTHORIZED WEB SECURITY SCAN")
print("=" * 60)
print(f"Target: {TARGET}")
print(f"Port: {PORT}")
print("Scope validation successful.")
print()


# --- RUN WEB SECURITY SCANNERS ---

scanner_results = []

scanners = [
    ("HTTP Security Header Scanner", scan_http_headers),
    ("HTTP Method Enumeration", enumerate_http_methods),
    ("Cookie Security Analyzer", analyze_cookie_security)
]

for scanner_name, scanner_function in scanners:

    # Validate scope before each scanner executes
    check_scope(TARGET, PORT)

    print(f"Running: {scanner_name}")

    try:
        result = scanner_function(
            target=TARGET,
            port=PORT
        )

        # Convert JSON string into a Python dictionary
        parsed_result = json.loads(result)

        scanner_results.append(parsed_result)

        print(json.dumps(parsed_result, indent=2))

    except PermissionError:
        raise

    except Exception as exc:
        error_result = {
            "scanner": scanner_name,
            "target": f"http://{TARGET}:{PORT}/",
            "error": str(exc)
        }

        scanner_results.append(error_result)

        print(json.dumps(error_result, indent=2))

    print("-" * 60)


# --- SAVE SCANNER RESULTS ---

output_path = Path("web_scan_results.json")

output_path.write_text(
    json.dumps(scanner_results, indent=2),
    encoding="utf-8"
)

print()
print("=" * 60)
print("SCANNING COMPLETE")
print("=" * 60)
print(f"Scanners executed: {len(scanner_results)}")
print(f"Results saved to: {output_path.resolve()}")


AUTHORIZED WEB SECURITY SCAN
Target: 127.0.0.1
Port: 3000
Scope validation successful.

Running: HTTP Security Header Scanner
{
  "scanner": "HTTP Security Header Scanner",
  "target": "http://127.0.0.1:3000/",
  "http_status": 200,
  "findings": [
    {
      "header": "Content-Security-Policy",
      "status": "MISSING",
      "observed_value": null,
      "finding": "Content-Security-Policy was not present in the response."
    },
    {
      "header": "X-Frame-Options",
      "status": "PRESENT",
      "observed_value": "SAMEORIGIN",
      "finding": "X-Frame-Options was present."
    },
    {
      "header": "X-Content-Type-Options",
      "status": "PRESENT",
      "observed_value": "nosniff",
      "finding": "X-Content-Type-Options was present."
    },
    {
      "header": "Referrer-Policy",
      "status": "MISSING",
      "observed_value": null,
      "finding": "Referrer-Policy was not present in the response."
    },
    {
      "header": "Permissions-Policy",
      "statu